<a href="https://colab.research.google.com/github/MarkowitzMx/Programacion-para-analitica-descriptiva-y-predictiva-2026/blob/main/Sesion10_274690.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo:** Cristobal Lemus Rendon

**Matrícula:** 274690

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [16]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [17]:
# Trabajamos sobre una COPIA para no afectar df_marketing, que usan las demás actividades
df_marketing_renombrado = df_marketing.copy()

# Paso 1: unificamos mayúsculas/minúsculas en todos los nombres de columna.
# Esto ya resuelve casos como "Year_Birth" -> "year_birth", pero deja nombres
# poco claros como "mntwines" o "kidhome" tal cual (sin separadores internos).
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower()

print("Columnas después de aplicar .str.lower():")
print(list(df_marketing_renombrado.columns))

# Paso 2: renombramos manualmente los nombres que la técnica automática (.lower())
# no deja perfectos, porque son abreviaturas o palabras pegadas sin separador.
# Elegimos nombres en español, descriptivos y con guion bajo, para mantener consistencia.
df_marketing_renombrado = df_marketing_renombrado.rename(columns={
    'year_birth': 'anio_nacimiento',   # abreviado en inglés -> nombre claro en español
    'kidhome':    'ninos_en_casa',     # "kidhome" no separa las dos palabras
    'teenhome':   'adolescentes_en_casa',
    'mntwines':   'monto_vinos',       # "mnt" = "amount"; poco intuitivo sin contexto
})

print("\nColumnas después del rename manual (muestra de las modificadas):")
print([c for c in df_marketing_renombrado.columns
       if c in ['anio_nacimiento', 'ninos_en_casa', 'adolescentes_en_casa', 'monto_vinos']])

print("\nListado completo de columnas final:")
print(list(df_marketing_renombrado.columns))


Columnas después de aplicar .str.lower():
['id', 'year_birth', 'education', 'marital_status', 'income', 'kidhome', 'teenhome', 'dt_customer', 'recency', 'mntwines', 'mntfruits', 'mntmeatproducts', 'mntfishproducts', 'mntsweetproducts', 'mntgoldprods', 'numdealspurchases', 'numwebpurchases', 'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth', 'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1', 'acceptedcmp2', 'complain', 'z_costcontact', 'z_revenue', 'response']

Columnas después del rename manual (muestra de las modificadas):
['anio_nacimiento', 'ninos_en_casa', 'adolescentes_en_casa', 'monto_vinos']

Listado completo de columnas final:
['id', 'anio_nacimiento', 'education', 'marital_status', 'income', 'ninos_en_casa', 'adolescentes_en_casa', 'dt_customer', 'recency', 'monto_vinos', 'mntfruits', 'mntmeatproducts', 'mntfishproducts', 'mntsweetproducts', 'mntgoldprods', 'numdealspurchases', 'numwebpurchases', 'numcatalogpurchases', 'numstorepurchases', 'numwebvis

---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [18]:
# Contamos los nulos ANTES de convertir, para poder comparar contra los de después
nulos_antes = df_netflix['date_added'].isnull().sum()
print("Tipo de dato ANTES de convertir:", df_netflix['date_added'].dtype)
print("Valores nulos en 'date_added' ANTES de convertir:", nulos_antes)

# format='mixed' le indica a pandas que, fila por fila, detecte automáticamente
# si el valor viene como "14-Aug-20" (formato corto) o como " August 4, 2017"
# (formato largo, con espacio inicial). Sin 'mixed', pd.to_datetime fallaría
# al encontrar dos formatos distintos en la misma columna.
df_netflix['date_added'] = pd.to_datetime(df_netflix['date_added'], format='mixed')

# Verificamos el tipo de dato resultante con .dtypes
print("\nTipo de dato DESPUÉS de convertir:")
print(df_netflix.dtypes['date_added'])

# Contamos los nulos DESPUÉS de convertir
nulos_despues = df_netflix['date_added'].isnull().sum()
print("\nValores nulos en 'date_added' DESPUÉS de convertir:", nulos_despues)

# Comparación: si format='mixed' funcionó bien, no deberían generarse nulos nuevos
# (los nulos originales, si existían, se mantienen; no debería haber más).
print(f"\n¿Se generaron nulos nuevos por la conversión? "
      f"{'Sí (revisar valores no reconocidos)' if nulos_despues > nulos_antes else 'No, la conversión fue limpia'}")


Tipo de dato ANTES de convertir: object
Valores nulos en 'date_added' ANTES de convertir: 10

Tipo de dato DESPUÉS de convertir:
datetime64[ns]

Valores nulos en 'date_added' DESPUÉS de convertir: 10

¿Se generaron nulos nuevos por la conversión? No, la conversión fue limpia


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [19]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [20]:
# Duplicados exactos: compara TODAS las columnas del dataframe, fila por fila.
# Una fila se marca como duplicada si es idéntica a otra fila anterior en TODOS los campos.
duplicados_exactos = df_marketing_dup.duplicated().sum()
print("Duplicados exactos (todas las columnas):", duplicados_exactos)

# Duplicados por 'ID': como el ID es único por cliente en este dataset,
# cualquier ID que se repita es, por definición, un cliente duplicado
# (sin importar si el resto de las columnas también coinciden).
duplicados_por_id = df_marketing_dup.duplicated(subset='ID').sum()
print("Duplicados por columna 'ID':", duplicados_por_id)

print(f"\n¿Coinciden ambos conteos? {'Sí' if duplicados_exactos == duplicados_por_id else 'No'}")
print("(Coinciden porque las 2 filas inyectadas son copias completas de clientes ya existentes,")
print(" así que duplican tanto el ID como el resto de las columnas al mismo tiempo).")

# Eliminamos los duplicados. drop_duplicates() por defecto conserva la PRIMERA
# aparición de cada fila y elimina las repeticiones posteriores.
df_marketing_sin_dup = df_marketing_dup.drop_duplicates()

print("\nFilas antes de eliminar duplicados:", len(df_marketing_dup))
print("Filas después de eliminar duplicados:", len(df_marketing_sin_dup))
print("Filas eliminadas:", len(df_marketing_dup) - len(df_marketing_sin_dup))


Duplicados exactos (todas las columnas): 2
Duplicados por columna 'ID': 2

¿Coinciden ambos conteos? Sí
(Coinciden porque las 2 filas inyectadas son copias completas de clientes ya existentes,
 así que duplican tanto el ID como el resto de las columnas al mismo tiempo).

Filas antes de eliminar duplicados: 2242
Filas después de eliminar duplicados: 2240
Filas eliminadas: 2


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [21]:
# .isnull() marca True/False por celda; .sum() por columna cuenta los True (nulos) de cada una
nulos_por_columna = df_netflix.isnull().sum()
print("Valores faltantes por columna (todas):")
print(nulos_por_columna)

# Filtramos para mostrar solo las columnas que sí tienen al menos un nulo (más legible)
print("\nSolo columnas con al menos un valor faltante:")
print(nulos_por_columna[nulos_por_columna > 0])

# .isnull().any(axis=1) recorre cada FILA y devuelve True si ALGUNA columna
# de esa fila tiene un valor nulo (axis=1 = "a lo largo de las columnas, por fila")
filas_con_nulos = df_netflix.isnull().any(axis=1)

total_filas_con_nulos = filas_con_nulos.sum()
print("\nTotal de filas con al menos un valor faltante:", total_filas_con_nulos)
print("Total de filas en el dataset:", len(df_netflix))
print(f"Porcentaje de filas afectadas: {total_filas_con_nulos / len(df_netflix) * 100:.2f}%")


Valores faltantes por columna (todas):
show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64

Solo columnas con al menos un valor faltante:
director      2389
cast           718
country        507
date_added      10
rating           7
dtype: int64

Total de filas con al menos un valor faltante: 2979
Total de filas en el dataset: 7787
Porcentaje de filas afectadas: 38.26%


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [22]:
total_filas = len(df_netflix)

# Completitud = porcentaje de valores que SÍ están presentes (no nulos) en cada columna.
# Si una columna no tiene nulos, su completitud es 100%.
completitud = (1 - df_netflix.isnull().sum() / total_filas) * 100

# Ordenamos de menor a mayor completitud para identificar rápido la columna más problemática
completitud_ordenada = completitud.sort_values()

print("Completitud por columna (%), ordenada de la más incompleta a la más completa:")
print(completitud_ordenada.round(2))

columna_mas_incompleta = completitud_ordenada.index[0]
valor_mas_bajo = completitud_ordenada.iloc[0]

# Respuesta en una línea:
print(f"\nRespuesta: la columna con menor completitud es '{columna_mas_incompleta}' "
      f"con {valor_mas_bajo:.2f}% (es decir, la que más valores nulos concentra).")


Completitud por columna (%), ordenada de la más incompleta a la más completa:
director         69.32
cast             90.78
country          93.49
date_added       99.87
rating           99.91
title           100.00
show_id         100.00
type            100.00
release_year    100.00
duration        100.00
listed_in       100.00
description     100.00
dtype: float64

Respuesta: la columna con menor completitud es 'director' con 69.32% (es decir, la que más valores nulos concentra).


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [23]:
# .value_counts() cuenta cuántas veces aparece cada valor único en la columna
conteo_estado_civil = df_marketing['Marital_Status'].value_counts()
print("Conteo de valores en 'Marital_Status':")
print(conteo_estado_civil)

# Aislamos los tres valores "raros" para verlos con claridad
valores_raros = conteo_estado_civil.loc[conteo_estado_civil.index.isin(['Alone', 'Absurd', 'YOLO'])]
print("\nValores que claramente son errores de captura o categorías no estándar:")
print(valores_raros)

# Decisión y justificación (una línea):
# - "Alone" se RECLASIFICARÍA a "Single", porque describe la misma condición civil real
#   (vivir sin pareja) y no aporta información nueva al segmentar como categoría aparte.
# - "Absurd" y "YOLO" se ELIMINARÍAN (o se marcarían como nulo), porque no describen un
#   estado civil válido, sino respuestas de broma o error de captura en el formulario.
print("\nDecisión: reclasificar 'Alone' -> 'Single'; eliminar/marcar como nulo "
      "'Absurd' y 'YOLO' por no ser estados civiles válidos ni tener un equivalente claro.")


Conteo de valores en 'Marital_Status':
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

Valores que claramente son errores de captura o categorías no estándar:
Marital_Status
Alone     3
Absurd    2
YOLO      2
Name: count, dtype: int64

Decisión: reclasificar 'Alone' -> 'Single'; eliminar/marcar como nulo 'Absurd' y 'YOLO' por no ser estados civiles válidos ni tener un equivalente claro.


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [24]:
# .str.match(patron) evalúa, para cada valor de la columna, si CUMPLE el patrón
# desde el inicio de la cadena. El patrón r'^s\d+$' significa:
#   ^      -> inicio de la cadena
#   s      -> la letra "s" literal
#   \d+    -> uno o más dígitos
#   $      -> fin de la cadena (no permite nada después de los dígitos)
cumple_patron = df_netflix['show_id'].str.match(r'^s\d+$')

total_valores = len(cumple_patron)
valores_que_cumplen = cumple_patron.sum()
porcentaje_cumplimiento = valores_que_cumplen / total_valores * 100

print("Total de valores evaluados:", total_valores)
print("Valores que cumplen el patrón 's' + dígitos:", valores_que_cumplen)
print(f"Porcentaje de cumplimiento: {porcentaje_cumplimiento:.2f}%")

# Mostramos, si existen, los valores que NO cumplen el patrón (para inspección manual)
valores_no_cumplen = df_netflix.loc[~cumple_patron, 'show_id']
print(f"\nCantidad de valores que NO cumplen el patrón: {len(valores_no_cumplen)}")
if len(valores_no_cumplen) > 0:
    print("Ejemplos de valores fuera de patrón:")
    print(valores_no_cumplen.head())
else:
    print("Todos los valores de 'show_id' cumplen la convención esperada.")


Total de valores evaluados: 7787
Valores que cumplen el patrón 's' + dígitos: 7787
Porcentaje de cumplimiento: 100.00%

Cantidad de valores que NO cumplen el patrón: 0
Todos los valores de 'show_id' cumplen la convención esperada.


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [25]:
# .describe() sobre una sola columna numérica muestra count, mean, std, min, 25%, 50%, 75%, max
print("Estadísticas descriptivas de 'Year_Birth':")
print(df_marketing[['Year_Birth']].describe())

# El valor mínimo (min) de Year_Birth suele ser un año muy antiguo (por ejemplo, 1893 o 1900),
# lo cual implicaría una edad superior a 120 años en la fecha del estudio: no tiene sentido
# para una encuesta real de marketing dirigida a clientes actuales.

# Filtramos las filas con los años de nacimiento más antiguos (usamos un umbral arbitrario
# pero razonable: antes de 1940, es decir, clientes que tendrían más de ~85 años)
filas_sospechosas = df_marketing[df_marketing['Year_Birth'] < 1940]
print("\nFilas con los años de nacimiento más antiguos (posibles errores de captura):")
print(filas_sospechosas[['ID', 'Year_Birth']])

# Decisión (una línea):
print("\nDecisión: sí, se consideran errores de captura, ya que implican edades poco "
      "plausibles (más de 85-130 años) para la población típica de una encuesta de marketing.")


Estadísticas descriptivas de 'Year_Birth':
        Year_Birth
count  2240.000000
mean   1968.805804
std      11.984069
min    1893.000000
25%    1959.000000
50%    1970.000000
75%    1977.000000
max    1996.000000

Filas con los años de nacimiento más antiguos (posibles errores de captura):
        ID  Year_Birth
192   7829        1900
239  11004        1893
339   1150        1899

Decisión: sí, se consideran errores de captura, ya que implican edades poco plausibles (más de 85-130 años) para la población típica de una encuesta de marketing.


---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [26]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [27]:
# Paso 1 — Ajuste de tipos
# Revisamos los tipos de dato actuales; 'Income' debería ser numérica pero quedó
# como 'object' porque en la celda anterior se inyectó el texto 'sesenta mil'.
print("Tipos de dato ANTES de la corrección:")
print(df_practica.dtypes)

# pd.to_numeric(..., errors='coerce') intenta convertir cada valor a número;
# los valores que NO se pueden convertir (como el texto 'sesenta mil') se
# transforman en NaN en lugar de generar un error que detenga la ejecución.
df_practica['Income'] = pd.to_numeric(df_practica['Income'], errors='coerce')

print("\nTipos de dato DESPUÉS de la corrección:")
print(df_practica.dtypes)

print("\nValor en la fila 5 de 'Income' después de la conversión "
      "(antes era el texto 'sesenta mil'):", df_practica.loc[5, 'Income'])


Tipos de dato ANTES de la corrección:
ID                      int64
Year_Birth              int64
Education              object
Marital_Status         object
Income                 object
Kidhome                 int64
Teenhome                int64
Dt_Customer            object
Recency                 int64
MntWines                int64
MntFruits               int64
MntMeatProducts         int64
MntFishProducts         int64
MntSweetProducts        int64
MntGoldProds            int64
NumDealsPurchases       int64
NumWebPurchases         int64
NumCatalogPurchases     int64
NumStorePurchases       int64
NumWebVisitsMonth       int64
AcceptedCmp3            int64
AcceptedCmp4            int64
AcceptedCmp5            int64
AcceptedCmp1            int64
AcceptedCmp2            int64
Complain                int64
Z_CostContact           int64
Z_Revenue               int64
Response                int64
dtype: object

Tipos de dato DESPUÉS de la corrección:
ID                       int64
Year_B

In [28]:
# Paso 2 — Duplicados
# Contamos las filas duplicadas (debería detectar la fila inyectada como copia de otra)
duplicados_practica = df_practica.duplicated().sum()
print("Filas duplicadas encontradas:", duplicados_practica)

# Eliminamos los duplicados y reiniciamos el índice para que quede limpio y consecutivo
filas_antes = len(df_practica)
df_practica = df_practica.drop_duplicates().reset_index(drop=True)
filas_despues = len(df_practica)

print(f"Filas antes de eliminar duplicados: {filas_antes}")
print(f"Filas después de eliminar duplicados: {filas_despues}")


Filas duplicadas encontradas: 1
Filas antes de eliminar duplicados: 16
Filas después de eliminar duplicados: 15


In [29]:
# Paso 3 — Valores faltantes
# Contamos los nulos por columna; esto debe incluir el nulo generado en el Paso 1
# (la fila donde estaba el texto 'sesenta mil' en 'Income', ahora convertido en NaN)
nulos_practica = df_practica.isnull().sum()
print("Valores faltantes por columna (solo las que tienen al menos uno):")
print(nulos_practica[nulos_practica > 0])

print("\nRecordatorio: el nulo en 'Income' corresponde al valor de tipo incorrecto "
      "('sesenta mil') que se convirtió a NaN durante el ajuste de tipos del Paso 1.")


Valores faltantes por columna (solo las que tienen al menos uno):
Income    1
dtype: int64

Recordatorio: el nulo en 'Income' corresponde al valor de tipo incorrecto ('sesenta mil') que se convirtió a NaN durante el ajuste de tipos del Paso 1.


In [30]:
# Paso 4 — Exploración categórica
# Revisamos los valores únicos de 'Marital_Status' en esta muestra específica
valores_unicos_estado_civil = df_practica['Marital_Status'].unique()
print("Valores únicos en 'Marital_Status' (muestra de la práctica integradora):")
print(valores_unicos_estado_civil)

# Aplicamos el mismo criterio de la Actividad 6: si aparecen categorías no válidas
# (Alone, Absurd, YOLO) en esta muestra, se normalizarían con la misma lógica.
categorias_no_validas = [v for v in valores_unicos_estado_civil if v in ['Alone', 'Absurd', 'YOLO']]
if categorias_no_validas:
    print(f"\nSe encontraron categorías no válidas en esta muestra: {categorias_no_validas}")
    print("Se normalizarían con el mismo criterio de la Actividad 6 "
          "('Alone' -> 'Single'; 'Absurd' y 'YOLO' se eliminarían/marcarían como nulo).")
else:
    print("\nEn esta muestra en particular no aparecen categorías no válidas de "
          "'Marital_Status', por lo que no se requiere normalización adicional aquí.")


Valores únicos en 'Marital_Status' (muestra de la práctica integradora):
['Together' 'Single' 'Married' 'Divorced']

En esta muestra en particular no aparecen categorías no válidas de 'Marital_Status', por lo que no se requiere normalización adicional aquí.


**Tu reporte de profiling:**  En la muestra de 15 clientes (`df_practica`) se identificaron y corrigieron cuatro problemas típicos de calidad de datos:

1. **Tipos de datos:** un error de tipo en `Income`, donde el texto "sesenta mil" impedía tratar la columna como numérica y se resolvió con `pd.to_numeric(errors='coerce')`, generando un valor nulo controlado; (
2. **Duplicados:** una fila duplicada exacta, eliminada con `.drop_duplicates()`;
3. **Valores faltantes:**  un valor faltante en `Income` (el mismo generado en el paso de tipos), confirmado con `.isnull().sum()`; y
4. **Categorías:** una revisión de `Marital_Status` que, en esta muestra puntual, no mostró categorías inválidas, aunque de aparecer se normalizarían con el mismo criterio usado en la Actividad 6.


En conjunto, el dataset quedó con tipos correctos, sin duplicados y con los nulos identificados y cuantificados, listo para un análisis posterior.






